In [ ]:
import numpy as np
from numba import njit, prange, cuda
import time
import math
import matplotlib.pyplot as plt
import taichi as ti

width, height = 8000, 8000
xmin, xmax = -0.2, 0.2
ymin, ymax = -1.1, -0.7
max_iter = 255
escape_radius_sq = 100.0
DTYPES = (np.float64, np.float32)

def mandelbrot_cpu():
    result = []
    for y in range(height):
        row = []
        imag = ymin + (y / (height - 1)) * (ymax - ymin)
        for x in range(width):
            real = xmin + (x / (width - 1)) * (xmax - xmin)
            c = complex(real, imag)
            z = 0
            iter_count = 0
            while abs(z)**2 <= escape_radius_sq and iter_count < max_iter:
                z = z * z + c
                iter_count += 1
                
            if iter_count < max_iter:
                z_norm = abs(z)
                nu = iter_count + 1 - math.log2(math.log2(z_norm))
                row.append(nu)
            else:
                row.append(float(max_iter))
        result.append(row)
    return result

def mandelbrot_numpy(dtype):
    x = np.linspace(xmin, xmax, width, dtype=dtype)
    y = np.linspace(ymin, ymax, height, dtype=dtype)
    X, Y = np.meshgrid(x, y)
    C = X + 1j * Y
    Z = np.zeros_like(C)
    assert Z.dtype.itemsize == 2 * np.dtype(dtype).itemsize, "NumPy po cichu zmienił precyzję"
    mandelbrot = np.full(C.shape, max_iter, dtype=np.float32)

    active = np.ones(C.shape, dtype=bool)

    for i in range(max_iter):
        if not active.any():
            break

        Z[active] = Z[active] ** 2 + C[active]
        escaped = np.abs(Z[active])**2 > escape_radius_sq

        if escaped.any():
            just_escaped = active.copy()
            just_escaped[active] = escaped

            Z_esc = Z[just_escaped]
            nu = (i + 2) - np.log2(np.log2(np.abs(Z_esc)))
            mandelbrot[just_escaped] = nu

            active[just_escaped] = False

    return mandelbrot

# Numba i CUDA: stałe typu dtype(...) wymuszają precyzję obliczeń. Bez nich literały 0.0 i 2.0
# (oraz współrzędne z globalnych float64) po cichu awansowałyby wszystko do float64.
def make_mandelbrot_numba(dtype):
    f = dtype

    @njit
    def mandelbrot_pixel(real, imag):
        z_real, z_imag = f(0.0), f(0.0)
        for i in range(max_iter):
            z2_real = z_real * z_real - z_imag * z_imag + real
            z2_imag = f(2.0) * z_real * z_imag + imag
            z_real, z_imag = z2_real, z2_imag

            z_sq = z_real * z_real + z_imag * z_imag
            if z_sq > f(escape_radius_sq):
                z_norm = math.sqrt(z_sq)
                return f(i + 2) - math.log2(math.log2(z_norm))
        return f(max_iter)

    @njit(parallel=True)
    def mandelbrot_numba():
        result = np.zeros((height, width), dtype=np.float32)
        for y in prange(height):
            imag = f(ymin + (y / (height - 1)) * (ymax - ymin))
            for x in range(width):
                real = f(xmin + (x / (width - 1)) * (xmax - xmin))
                result[y, x] = mandelbrot_pixel(real, imag)
        return result

    return mandelbrot_numba

def make_mandelbrot_kernel(dtype):
    f = dtype

    @cuda.jit
    def mandelbrot_kernel(result):
        x, y = cuda.grid(2)
        if x < width and y < height:
            real = f(xmin + (x / (width - 1)) * (xmax - xmin))
            imag = f(ymin + (y / (height - 1)) * (ymax - ymin))
            z_real, z_imag = f(0.0), f(0.0)
            for i in range(max_iter):
                z2_real = z_real * z_real - z_imag * z_imag + real
                z2_imag = f(2.0) * z_real * z_imag + imag
                z_real, z_imag = z2_real, z2_imag

                z_sq = z_real * z_real + z_imag * z_imag
                if z_sq > f(escape_radius_sq):
                    z_norm = math.sqrt(z_sq)
                    result[y, x] = f(i + 2) - math.log2(math.log2(z_norm))
                    return
            result[y, x] = f(max_iter)

    return mandelbrot_kernel

def mandelbrot_cuda(kernel, d_result):
    threadsperblock = (16, 16)
    blockspergrid = (math.ceil(width / threadsperblock[0]), math.ceil(height / threadsperblock[1]))

    kernel[blockspergrid, threadsperblock](d_result)
    cuda.default_stream().synchronize()

def mandelbrot_taichi(arch_name, dtype):
    arch = ti.cuda if arch_name == "gpu" else ti.cpu
    ti.reset()
    ti.init(arch=arch, default_fp=ti.f32 if dtype == np.float32 else ti.f64)
    if ti.lang.impl.current_cfg().arch != arch:
        raise RuntimeError(f"Taichi nie uruchomił architektury {arch_name} (brak CUDA?)")

    pixels = ti.field(dtype=ti.f32, shape=(height, width))

    @ti.kernel
    def render():
        for y, x in pixels:
            real = xmin + (x / (width - 1)) * (xmax - xmin)
            imag = ymin + (y / (height - 1)) * (ymax - ymin)
            z_real = 0.0
            z_imag = 0.0
            escaped = False
            for k in range(max_iter):
                z2_real = z_real * z_real - z_imag * z_imag + real
                z2_imag = 2.0 * z_real * z_imag + imag
                z_real, z_imag = z2_real, z2_imag

                z_sq = z_real * z_real + z_imag * z_imag
                if z_sq > escape_radius_sq:
                    z_norm = ti.math.sqrt(z_sq)
                    pixels[y, x] = ti.cast((k + 2) - ti.math.log2(ti.math.log2(z_norm)), ti.f32)
                    escaped = True
                    break
            if not escaped:
                pixels[y, x] = ti.cast(max_iter, ti.f32)

    start = time.perf_counter()
    render()
    ti.sync()
    comp_exec_time = time.perf_counter() - start

    start = time.perf_counter()
    render()
    ti.sync()
    exec_time = time.perf_counter() - start
    comp_time = comp_exec_time - exec_time

    return pixels.to_numpy(), comp_time, exec_time

def measure(fn):
    """Zwraca (wynik, czas kompilacji, czas wykonania): kompilacja = pierwsze wywołanie minus drugie."""
    start = time.perf_counter()
    fn()
    first_call = time.perf_counter() - start

    start = time.perf_counter()
    result = fn()
    exec_time = time.perf_counter() - start
    return result, first_call - exec_time, exec_time

def prepare_img(img_floats, style="neon"):
    mask = img_floats < max_iter
    
    t = np.zeros_like(img_floats)
    t[mask] = img_floats[mask] / 40.0 
    
    if style == "fire":
        A = np.array([0.5, 0.5, 0.5])
        B = np.array([0.5, 0.5, 0.5])
        C = np.array([1.0, 1.0, 0.5])
        D = np.array([0.00, 0.15, 0.20])
    elif style == "neon":
        A = np.array([0.5, 0.5, 0.5])
        B = np.array([0.5, 0.5, 0.5])
        C = np.array([1.0, 1.0, 1.0])
        D = np.array([0.263, 0.416, 0.557])
    else:
        A, B, C, D = np.ones(3)*0.5, np.ones(3)*0.5, np.ones(3), np.zeros(3)

    r = np.zeros_like(img_floats)
    g = np.zeros_like(img_floats)
    b = np.zeros_like(img_floats)
    
    r[mask] = A[0] + B[0] * np.cos(2 * np.pi * (C[0] * t[mask] + D[0]))
    g[mask] = A[1] + B[1] * np.cos(2 * np.pi * (C[1] * t[mask] + D[1]))
    b[mask] = A[2] + B[2] * np.cos(2 * np.pi * (C[2] * t[mask] + D[2]))
    
    rgb = np.dstack((r, g, b))
    rgb = np.clip(rgb, 0.0, 1.0) * 255.0
    return rgb.astype(np.uint8)

def run_numpy(dtype):
    start = time.perf_counter()
    image = mandelbrot_numpy(dtype)
    return image, 0.0, time.perf_counter() - start

def run_numba(dtype):
    return measure(make_mandelbrot_numba(dtype))

def run_cuda(dtype):
    kernel = make_mandelbrot_kernel(dtype)
    d_result = cuda.device_array((height, width), dtype=np.float32)
    _, comp_time, exec_time = measure(lambda: mandelbrot_cuda(kernel, d_result))
    return d_result.copy_to_host(), comp_time, exec_time

METHODS = {
    "Numpy": run_numpy,
    "Numba": run_numba,
    "CUDA": run_cuda,
    "Taichi CPU": lambda dtype: mandelbrot_taichi("cpu", dtype),
    "Taichi GPU": lambda dtype: mandelbrot_taichi("gpu", dtype),
}

# timings[(metoda, dtype)] = (czas kompilacji, czas wykonania); czysty Python ma tylko float64
timings = {}
images = {}
mismatch = {}

print("Liczenie CPU...")
start = time.perf_counter()
cpu_rows = mandelbrot_cpu()
timings[("CPU", np.float64)] = (0.0, time.perf_counter() - start)
images["CPU"] = np.array(cpu_rows, dtype=np.float32)
del cpu_rows

for name, run in METHODS.items():
    for dtype in DTYPES:
        print(f"Liczenie {name} ({dtype.__name__})...")
        image, comp_time, exec_time = run(dtype)
        timings[(name, dtype)] = (comp_time, exec_time)
        if dtype is np.float64:
            images[name] = image
        else:
            # różnica względem wyniku float64 tej samej metody: ile pikseli różni się o więcej niż 1 iterację
            diff = np.abs(image - images[name])
            mismatch[name] = (100.0 * np.mean(diff > 1.0), float(diff.max()))

def timing_line(name, dtype):
    if (name, dtype) not in timings:
        return f"{dtype.__name__}: N/A"
    comp_time, exec_time = timings[(name, dtype)]
    comp = "N/A" if name in ("CPU", "Numpy") else f"{comp_time:.4f} s"
    return f"{dtype.__name__}: Wykonanie {exec_time:.4f} s | Kompilacja {comp} | Suma {exec_time + comp_time:.4f} s"

fig, axs = plt.subplots(3, 2, figsize=(14, 20))
axs = axs.ravel()

panels = [("CPU", "CPU"), ("Numpy", "Numpy"), ("Numba", "Numba"), ("CUDA", "CUDA"),
          ("Taichi CPU", "Taichi CPU"), ("Taichi GPU", "Taichi GPU")]

for ax, (name, label) in zip(axs, panels):
    ax.imshow(prepare_img(images[name]), extent=[xmin, xmax, ymin, ymax], origin="lower")
    ax.set_title("\n".join([label] + [timing_line(name, dtype) for dtype in DTYPES]), fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("wynik.png", dpi=100)
plt.show()

print()
print(f"{'Metoda':<12}{'dtype':<9}{'Wykonanie [s]':>15}{'Kompilacja [s]':>16}{'Suma [s]':>11}{'Różnica >1 iter. [%]':>23}{'Maks. różnica':>15}")
for name in ["CPU"] + list(METHODS):
    for dtype in DTYPES:
        if (name, dtype) not in timings:
            continue
        comp_time, exec_time = timings[(name, dtype)]
        diff = "" if name not in mismatch or dtype is np.float64 else f"{mismatch[name][0]:.4f}"
        maxdiff = "" if name not in mismatch or dtype is np.float64 else f"{mismatch[name][1]:.2f}"
        print(f"{name:<12}{dtype.__name__:<9}{exec_time:>15.4f}{comp_time:>16.4f}{exec_time + comp_time:>11.4f}{diff:>23}{maxdiff:>15}")
